In [4]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

ImportError: cannot import name 'display' from 'IPython.core.display' (/opt/anaconda3/lib/python3.13/site-packages/IPython/core/display.py)

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [17]:
# Librerías principales
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

RANDOM_STATE = 42
try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    english_stopwords = set(stopwords.words("english"))
except Exception:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    english_stopwords = set(ENGLISH_STOP_WORDS)

print("Environment ready.")

Environment ready.


- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [26]:
## Read Data for the Fraudulent Email Kaggle Challenge
possible_train_paths = [
    "kg_train.csv",
    "kg_train (1).csv",
    "/mnt/data/kg_train (1).csv"
]

possible_test_paths = [
    "kg_test.csv",
    "kg_test (1).csv",
    "/mnt/data/kg_test (1).csv"
]

train_path = next((path for path in possible_train_paths if os.path.exists(path)), None)
test_path = next((path for path in possible_test_paths if os.path.exists(path)), None)

if train_path is None:
    raise FileNotFoundError("No encuentro kg_train.csv. Pon el CSV en la misma carpeta del notebook.")

if test_path is None:
    raise FileNotFoundError("No encuentro kg_test.csv. Pon el CSV en la misma carpeta del notebook.")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())
display(test.head())

Train shape: (5964, 2)
Test shape: (5964, 1)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


,text
0,usiness is for the fact that the deceased man ...
1,They are happy to adjust to the afternoon. I a...
2,Lael Brainard was confirmed 78-19 this afterno...
3,H <hrod17@clintonemail.com>Friday March 26 201...
4,"n;""> Dear Good Friend,<br><br><br>I am happy t..."


In [27]:
print("Columnas train:", train.columns.tolist())
print("Columnas test:", test.columns.tolist())

print("\nValores nulos en train:")
print(train.isna().sum())

print("\nValores nulos en test:")
print(test.isna().sum())

print("\nDistribución de labels:")
print(train["label"].value_counts())
print(train["label"].value_counts(normalize=True))

print("\nDuplicados exactos en train:", train.duplicated().sum())
print("Textos duplicados en train:", train["text"].duplicated().sum())
print("Textos duplicados en test:", test["text"].duplicated().sum())

Columnas train: ['text', 'label']
Columnas test: ['text']

Valores nulos en train:
text     0
label    0
dtype: int64

Valores nulos en test:
text    0
dtype: int64

Distribución de labels:
label
0    3386
1    2578
Name: count, dtype: int64
label
0    0.56774
1    0.43226
Name: proportion, dtype: float64

Duplicados exactos en train: 651
Textos duplicados en train: 651
Textos duplicados en test: 642


### Let's divide the training and test set into two partitions

In [28]:
# Your code
train_unique = train.drop_duplicates(subset=["text"]).reset_index(drop=True)

X = train_unique["text"].astype(str)
y = train_unique["label"].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train interno:", X_train.shape)
print("Validación:", X_valid.shape)
print("Distribución validación:")
print(y_valid.value_counts())

# Convertimos las particiones X_train/X_valid en DataFrames
# para que las siguientes celdas del notebook funcionen con data_train y data_val

data_train = pd.DataFrame({
    "text": X_train,
    "label": y_train
}).reset_index(drop=True)

data_val = pd.DataFrame({
    "text": X_valid,
    "label": y_valid
}).reset_index(drop=True)

print("data_train:", data_train.shape)
print("data_val:", data_val.shape)

data_train.head()


Train interno: (4250,)
Validación: (1063,)
Distribución validación:
label
0    598
1    465
Name: count, dtype: int64
data_train: (4250, 2)
data_val: (1063, 2)


,text,label
0,Very much so. I am also making a few changes a...,0
1,Ok. How about 7? Does that work for him? Can I...,0
2,"$B?7$7$$%a!<%k%""%I%l%9$r$*CN$i$;$7$^$9?7$7$$%...",1
3,"Head of Operations,Commercial Finance ServiceU...",1
4,Mr George MboroConfidential Tel :++ 874-76-253...,1


## Data Preprocessing

In [29]:
import string

print("Punctuation:")
print(string.punctuation)

print("\nEjemplo de stopwords:")
print(list(english_stopwords)[100:110] if len(english_stopwords) > 110 else list(english_stopwords)[:10])

from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer("english")

Punctuation:
!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

Ejemplo de stopwords:
['did', 'their', 'mightn', 'just', "she'll", "wasn't", 'm', 'off', 'further', "don't"]


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [30]:
# Your code
# RESPUESTA: limpiar HTML, JavaScript, CSS, comentarios y etiquetas
data_train["preprocessed_text"] = ...
import re

def remove_html(text):
    text = str(text)

    # Remove inline JavaScript and CSS
    text = re.sub(r"<script.*?>.*?</script>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<style.*?>.*?</style>", " ", text, flags=re.DOTALL | re.IGNORECASE)

    # Remove HTML comments
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # Remove the remaining HTML tags
    text = re.sub(r"<.*?>", " ", text)

    return text

data_train["preprocessed_text"] = data_train["text"].apply(remove_html)
data_val["preprocessed_text"] = data_val["text"].apply(remove_html)

data_train[["text", "preprocessed_text"]].head()

,text,preprocessed_text
0,Very much so. I am also making a few changes a...,Very much so. I am also making a few changes a...
1,Ok. How about 7? Does that work for him? Can I...,Ok. How about 7? Does that work for him? Can I...
2,"$B?7$7$$%a!<%k%""%I%l%9$r$*CN$i$;$7$^$9?7$7$$%...","$B?7$7$$%a!<%k%""%I%l%9$r$*CN$i$;$7$^$9?7$7$$%..."
3,"Head of Operations,Commercial Finance ServiceU...","Head of Operations,Commercial Finance ServiceU..."
4,Mr George MboroConfidential Tel :++ 874-76-253...,Mr George MboroConfidential Tel :++ 874-76-253...


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [31]:
# Your code
# RESPUESTA: limpiar caracteres especiales, números, letras sueltas, espacios, prefijo b y pasar a minúsculas

def basic_text_cleaning(text):
    text = str(text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Remove all single characters
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text)

    # Remove single characters from the start
    text = re.sub(r"^[a-zA-Z]\s+", " ", text)

    # Remove prefixed b
    text = re.sub(r"^b\s+", " ", text)

    # Substitute multiple spaces with single space
    text = re.sub(r"\s+", " ", text)

    # Convert to lowercase
    text = text.lower().strip()

    return text

data_train["preprocessed_text"] = data_train["preprocessed_text"].apply(basic_text_cleaning)
data_val["preprocessed_text"] = data_val["preprocessed_text"].apply(basic_text_cleaning)

data_train[["text", "preprocessed_text"]].head()

,text,preprocessed_text
0,Very much so. I am also making a few changes a...,very much so am also making few changes as wel...
1,Ok. How about 7? Does that work for him? Can I...,ok how about does that work for him can call o...
2,"$B?7$7$$%a!<%k%""%I%l%9$r$*CN$i$;$7$^$9?7$7$$%...",a i r cn a i b lucie tn yahoo co jpbefore the ...
3,"Head of Operations,Commercial Finance ServiceU...",head of operations commercial finance serviceu...
4,Mr George MboroConfidential Tel :++ 874-76-253...,mr george mboroconfidential tel confidential f...


## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
# Your code

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [ ]:
# Your code

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [ ]:
# Your code

## Extra features

In [ ]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

## How would you create a Bag of Words with the CountVectorizer method?

In [ ]:
# Your code

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [24]:
# Your code
# RESPUESTA: TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf_vectorizer.transform(data_val["preprocessed_text"])

print("Shape X_train_tfidf:", X_train_tfidf.shape)
print("Shape X_val_tfidf:", X_val_tfidf.shape)

# Ejemplo de palabras aprendidas por el vectorizador
print("Primeras 20 features:")
print(tfidf_vectorizer.get_feature_names_out()[:20])

NameError: name 'data_train' is not defined

### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code